# 3. Look at your data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SMLCI/acia-core/blob/main/docs/tutorials/03_look_at_your_data.ipynb)

Before you segment anything, look at it. This notebook covers the three ways
`acia` shows you a sequence: an interactive viewer inside Jupyter, static
figures, and an annotated video with a scale bar and a clock.

In [ ]:
# On Colab (or any fresh environment) this installs acia.
# Locally, if you already have acia installed, it is a no-op.
try:
    import acia  # noqa: F401
except ImportError:
    %pip install -q acia

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

DATA_URL = "https://data.celltrackingchallenge.net/training-datasets/DIC-C2DH-HeLa.zip"
DATA_DIR = Path("data")
SEQUENCE = DATA_DIR / "DIC-C2DH-HeLa" / "01"

if not SEQUENCE.exists():
    DATA_DIR.mkdir(exist_ok=True)
    archive = DATA_DIR / "DIC-C2DH-HeLa.zip"
    print("downloading ~42 MB ...")
    urlretrieve(DATA_URL, archive)
    with ZipFile(archive) as zf:
        zf.extractall(DATA_DIR)
    archive.unlink()

print(SEQUENCE, "->", len(list(SEQUENCE.glob("*.tif"))), "frames")

## Load and shrink

We attach the dataset's documented calibration right away — 0.19 µm per pixel at
a 10 minute frame interval — because the scale bar and the timestamp need it.
[Tutorial 4](04_calibration_and_units.ipynb) goes into what else that buys you.

Then we subsample. 84 frames is not much, but the habit matters: everything below
gets ~8x cheaper for free.

In [ ]:
from acia import ureg
from acia.segm.open import open_sequence

full = open_sequence(SEQUENCE).position(0)
full = full.with_pixel_size(0.19 * ureg.micrometer).with_frame_interval(
    10 * ureg.minute
)

src = full[::10]  # every 10th frame

print(f"{len(full)} frames -> {len(src)} frames")
print("pixel size    ", src.pixel_size)
print("timepoints    ", src.timepoints)

Note that the calibration followed the slice: the underlying interval is 10
minutes, so `src.timepoints` steps in **100 minute** increments rather than
silently staying at 10.

## The interactive viewer

Every source in `acia` mixes in
{class}`~acia.notebook.JupyterVisualizationMixin`, so making the last expression
of a cell a source gives you a viewer with a frame slider and channel toggles:

```python
src
```

That is genuinely interactive — it needs a running kernel and `ipywidgets`, so it
works in JupyterLab and on Colab but cannot be captured in this static page. Try
it in your own session; it is the fastest way to scrub through a sequence.

The rest of this notebook uses static rendering, which works everywhere.

## A contact sheet

For a quick overview of the whole time course, matplotlib is hard to beat.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(3, 3, figsize=(9, 9))

for idx, ax in enumerate(axes.ravel()):
    ax.imshow(np.asarray(src[idx].raw).squeeze(), cmap="gray")
    ax.set_title(f"{src.timepoints[idx].to('hour'):~.1f}", fontsize=9)
    ax.axis("off")

fig.suptitle("DIC-C2DH-HeLa, every 10th frame")
fig.tight_layout()
plt.show()

## Grayscale to RGB

`to_rgb()` turns any source into a three-channel one. On a single-channel
sequence it simply replicates the channel; on a multi-channel sequence you can
map each channel to a colour, which is how fluorescence composites are built.

In [ ]:
rgb = src.to_rgb()

print("channels:", src.num_channels, "->", rgb.num_channels)
print("frame shape:", np.asarray(rgb[0].raw).shape)

# multi-channel example (not executed -- our sample has one channel):
#   composite = two_channel_src.to_rgb(colors={0: "green", 1: "magenta"})

## Scale bar and clock

{func}`~acia.viz.render_scalebar` and {func}`~acia.viz.render_time` each take a
source and return a **new source** with the annotation burned in. Because they
return sources, they chain — and because they are lazy, chaining them costs
nothing until frames are pulled.

The scale bar is specified in physical units, which is exactly why the
calibration mattered.

In [ ]:
from acia.viz import render_scalebar, render_time

annotated = render_scalebar(
    rgb,
    xy_position=(20, 460),
    size_of_pixel=src.pixel_size,
    bar_width=10 * ureg.micrometer,
    bar_height=2 * ureg.micrometer,
)
annotated = render_time(
    annotated,
    xy_position=(20, 20),
    timepoints=list(src.timepoints),
    time_format="{H:02}h {M:02}m",
)

plt.figure(figsize=(5, 5))
plt.imshow(np.asarray(annotated[len(annotated) // 2].raw))
plt.axis("off")
plt.title("annotated frame")
plt.show()

## Export a video

{func}`~acia.viz.render_video` writes the sequence to a file. It is the last step
of most workflows and, on long sequences, often the slowest — one more reason to
subsample while you are still iterating.

In [ ]:
from pathlib import Path

from acia.viz import render_video

render_video(annotated, "sequence.mp4", framerate=4)

print("wrote sequence.mp4:", Path("sequence.mp4").stat().st_size // 1024, "KiB")

In [ ]:
from IPython.display import Video

Video("sequence.mp4", embed=True, width=420)

For finer control over codec and quality there is
{class}`~acia.viz.VideoExporter2`, a context manager with ready-made presets:

```python
from acia.viz import VideoExporter2

with VideoExporter2.default_h264("out.mp4", framerate=10) as exporter:
    for frame in annotated:
        exporter.write(frame.raw)
```

`default_vp9`, `fast_vp9`, `default_h264` and `default_h265` cover the usual
trade-offs between file size, encoding time and browser support.

## What you learned

* Any source displays as an **interactive viewer** in Jupyter — just put it last
  in a cell.
* `to_rgb()` produces displayable three-channel frames, with per-channel colours
  for composites.
* {func}`~acia.viz.render_scalebar` and {func}`~acia.viz.render_time` return new
  sources, so annotations chain lazily; the scale bar is in physical units.
* {func}`~acia.viz.render_video` writes the result out, and
  {class}`~acia.viz.VideoExporter2` gives you codec control.

Next: [4. Calibration and units](04_calibration_and_units.ipynb) — why
`pixel_size` is worth setting, and what it does to your results.